# Phase 5.2 — Batch SAP-aware evaluation (round 2)
Candidate ranges are prefilled for three readable studies, bound to exact PDF hashes. Confirm body review and permission before sending excerpts to DeepSeek. No labels or shell pages enter extraction.

In [ ]:
from pathlib import Path
import sys, subprocess, importlib
REPO_URL = 'https://github.com/wohaofanaweishenme-cell/tfl-generation-agent.git'
BRANCH = 'generic-retrieval-v2'
ROOT = Path('/content/tfl-generic-retrieval-v2')
if not ROOT.exists():
    subprocess.check_call(['git','clone','--depth','1','--branch',BRANCH,REPO_URL,str(ROOT)])
else:
    current = subprocess.check_output(['git','-C',str(ROOT),'branch','--show-current'], text=True).strip()
    assert current == BRANCH, f'Wrong branch: {current}'
    subprocess.check_call(['git','-C',str(ROOT),'pull','--ff-only'])
subprocess.check_call([sys.executable,'-m','pip','install','-q','-e',f'{ROOT}[notebook,test]','sentence-transformers>=3,<6'])
SRC = (ROOT / 'src').resolve()
assert (SRC / 'tfl_agent/tools/sap_hybrid.py').is_file(), SRC
if str(SRC) in sys.path:
    sys.path.remove(str(SRC))
sys.path.insert(0, str(SRC))
importlib.invalidate_caches()
cached = sys.modules.get('tfl_agent')
if cached is not None and not Path(cached.__file__).resolve().is_relative_to(SRC):
    raise RuntimeError('Wrong tfl_agent checkout already imported. Restart session and run all.')
# A git pull updates files, not objects already imported in this runtime.
# Clear this project's package cache before importing the new checkout.
for name in list(sys.modules):
    if name == 'tfl_agent' or name.startswith('tfl_agent.'):
        del sys.modules[name]
# Remove stale compiled bytecode, including same-second source updates.
import shutil
for cache in SRC.joinpath('tfl_agent').rglob('__pycache__'):
    shutil.rmtree(cache)
importlib.invalidate_caches()
import tfl_agent
from tfl_agent.agent.sap_scope_pilot import run_scope_retrieval
assert Path(tfl_agent.__file__).resolve().is_relative_to(SRC)
print('Commit:', subprocess.check_output(['git','-C',str(ROOT),'rev-parse','HEAD'],text=True).strip())
print('Agent package loaded:', tfl_agent.__file__)
from tfl_agent.agent.phase4_pipeline import run_phase4
from tfl_agent.agent.phase5_evaluation import run_phase5
from tfl_agent.agent.challenge_batch import read_csv, select_cohort, preflight, run_batch, preferred_configs
assert callable(preferred_configs), 'Updated batch module did not load'
print('Phase 5.2 round 2 imports verified.')


## 1. Load Phase 5.1 benchmark from Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import zipfile, json
from datetime import datetime, timezone
SAP_PATH=Path('/content/drive/MyDrive/Colab Notebooks/agent/midterm demo/SAP_001_00.pdf')
available=sorted((SAP_PATH.parent/'Phase5_Outputs').glob('*.zip'))
if not available: raise FileNotFoundError('Run Notebook 09 first')
RUN_ID=datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
OUT=Path('/content/Phase52')/RUN_ID
OUT.mkdir(parents=True)
with zipfile.ZipFile(available[-1]) as z:
    import io, csv
    title_rows=list(csv.DictReader(io.StringIO(z.read('Title_Benchmark.csv').decode('utf-8-sig'))))
cohort=select_cohort(title_rows)
print('Planned cases:',len(cohort))

## 2. Local PDF and mapping preflight
Candidate page limits exclude registered Content List and shell pages. Review actual body before approval.

In [ ]:
import pandas as pd
from IPython.display import display
preflight_rows, mapping_rows=preflight(ROOT,cohort,OUT)
display(pd.DataFrame(preflight_rows)[['study_id','pdf_path','first_content_list_page','first_shell_page','max_body_page']])
display(pd.DataFrame(mapping_rows)[['study_id','output_id','stratum','type_conflict','mapping_review_status']])
display(pd.read_csv(OUT/'Page_Review.csv')[['study_id','pdf','page','words','statistical_terms','preview']])
print('Preflight outputs:',OUT)

## 3. Confirm selected SAP body ranges and API authorization
Candidate physical pages: ARV 10–30, OMO 8–29, SP11631 5–27. These were reviewed from page previews, not full-page inspection. Verify the original PDFs before setting BODY_RANGES_VERIFIED. Only enable AUTHORIZED_TO_SEND_SAP if permitted. Both switches are required; otherwise the notebook exports preflight only.

In [ ]:
BODY_RANGES_VERIFIED = False #@param {type:"boolean"}
AUTHORIZED_TO_SEND_SAP = False #@param {type:"boolean"}
CONFIGS=preferred_configs(preflight_rows,body_verified=BODY_RANGES_VERIFIED,api_authorized=AUTHORIZED_TO_SEND_SAP)
display(pd.DataFrame(CONFIGS)[['study_id','body_first','body_last','body_verified','api_authorized']])
selected_studies={c['study_id'] for c in CONFIGS}
selected=[c for c in mapping_rows if c['study_id'] in selected_studies]
print('Configured cases:',len(selected),'Type conflicts:',sum(c['type_conflict'] for c in selected))
llm=None
if BODY_RANGES_VERIFIED and AUTHORIZED_TO_SEND_SAP:
    from google.colab import userdata
    from tfl_agent.tools.llm_client import DeepSeekJSONClient
    key=userdata.get('DEEPSEEK_API_KEY')
    if not key: raise RuntimeError('Missing DEEPSEEK_API_KEY')
    llm=DeepSeekJSONClient(model='deepseek-flash',api_key=key)
    del key
else:
    print('Preflight only: both confirmation switches must be enabled for API execution.')

## 4. Run batch and evaluate
No configured studies means all cases are reported as skipped, not failures or successful matches.

In [ ]:
summary=run_batch(ROOT,cohort,title_rows,preflight_rows,mapping_rows,CONFIGS,OUT,llm=llm)
print(json.dumps(summary,indent=2))
for label in ('paired_completed_metrics','paired_type_consistent_metrics','paired_type_conflict_metrics'):
    print(label)
    if summary[label]: display(pd.DataFrame(summary[label]).T)
display(pd.DataFrame(json.loads((OUT/'Case_Status.json').read_text()))[['study_id','output_id','stratum','status','detail']])

## 5. Export audit ZIP
Save even after a partial run. Includes potentially confidential SAP text; share only with authorization.

In [ ]:
import shutil
from google.colab import files
ZIP_PATH=Path(shutil.make_archive(str(OUT),'zip',OUT))
destination=SAP_PATH.parent/'Phase52_Outputs'
destination.mkdir(parents=True,exist_ok=True)
saved=Path(shutil.copy2(ZIP_PATH,destination/ZIP_PATH.name))
print('Saved:',saved)
files.download(str(ZIP_PATH))